In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings("ignore")
df = pd.read_csv("fake_job_postings.csv")
print(df.shape)
print(df.head())
print(df.columns.tolist())
# The original column 'Recruiter Decision' is not in this dataset. We will use 'fraudulent' as the target variable.
# print(df["Recruiter Decision"].value_counts())
# print(df.groupby("Recruiter Decision")["AI Score (0-100)"].mean())

# The original features are not in this dataset. We need to select new features.
# features = ["Experience (Years)", "Projects Count", "Salary Expectation ($)", "AI Score (0-100)"]

# For demonstration, let's select some numerical and encoded categorical features from the new dataset.
# You might want to choose different features based on your analysis goals.
features = ['telecommuting', 'has_company_logo', 'has_questions'] # Example numerical features

le_employment_type = LabelEncoder()
le_experience = LabelEncoder()
le_education = LabelEncoder()
le_industry = LabelEncoder()
le_function = LabelEncoder()

df['employment_type_encoded'] = le_employment_type.fit_transform(df['employment_type'].fillna('None'))
df['required_experience_encoded'] = le_experience.fit_transform(df['required_experience'].fillna('None'))
df['required_education_encoded'] = le_education.fit_transform(df['required_education'].fillna('None'))
df['industry_encoded'] = le_industry.fit_transform(df['industry'].fillna('None'))
df['function_encoded'] = le_function.fit_transform(df['function'].fillna('None'))

features = features + ['employment_type_encoded', 'required_experience_encoded', 'required_education_encoded', 'industry_encoded', 'function_encoded']

X = df[features]
y = df["fraudulent"] # Changed target variable to 'fraudulent'

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
print(features)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(X_train.shape[0])
print(X_test.shape[0])
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, class_weight="balanced")
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
print(accuracy_score(y_test, y_pred))

# Map numerical labels to descriptive strings for target_names
class_names = ['Not Fraudulent', 'Fraudulent'] # Assuming 0 is Not Fraudulent and 1 is Fraudulent
print(classification_report(y_test, y_pred, target_names=class_names))
print(confusion_matrix(y_test, y_pred))
importance = pd.DataFrame({"Feature": features, "Importance": model.feature_importances_}).sort_values("Importance", ascending=False)
print(importance)

sample = X_test.iloc[0:1]
sample_scaled = scaler.transform(sample)
prediction = model.predict(sample_scaled)[0]
probability = model.predict_proba(sample_scaled)[0]

print(sample)
print(le_target.inverse_transform([prediction])[0])
print(probability[1])
print(probability[0])

(1200, 18)
   job_id               title           location  department  salary_range  \
0       1   Software Engineer  San Francisco, CA     Finance  80000-120000   
1       2   Software Engineer       New York, NY         NaN   40000-60000   
2       3    Business Analyst       New York, NY  Operations  80000-120000   
3       4  Research Associate  San Francisco, CA         NaN   40000-60000   
4       5    Graphic Designer        Seattle, WA       Sales   30000-45000   

                                     company_profile  \
0  Join our growing team of professionals dedicat...   
1  We believe in creating a positive work environ...   
2  We are a leading technology company focused on...   
3  Join our growing team of professionals dedicat...   
4  Join our growing team of professionals dedicat...   

                                         description  \
0  The ideal candidate will have strong analytica...   
1  The ideal candidate will have strong analytica...   
2  We are looki